<a href="https://colab.research.google.com/github/andrew-veriga/Titans_jax/blob/main/colabs/Precompute_Activations_Gemma3_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemma-3-1B Activation Precomputation (GPU T4)

Этот ноутбук предназначен для предварительного вычисления активаций (hidden states) перед слоем Titans (слой 23) модели Gemma-3-1B. Это позволяет значительно ускорить обучение TitansBlock, так как графу JAX не нужно просчитывать первые 23 слоя Gemma при каждом шаге.

**Особенности:**
- Оптимизировано для GPU T4 (Colab Free/Pro).
- Автоматическая выгрузка шардов в HuggingFace Hub.
- Поддержка докачки (resume) при обрыве сессии.
- Автоматическая конвертация в Parquet с ZSTD-сжатием (объединяет 10 шардов в один файл).

In [ ]:
# 1. Установка зависимостей
# !pip install -q --upgrade "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install -q git+https://github.com/google-deepmind/gemma.git
!pip install -q flax==0.12.5 optax==0.2.6 typeguard==4.4.1 datasets ml_dtypes huggingface_hub orbax-checkpoint pyarrow

In [ ]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
hf_token

In [ ]:
# 2. Аутентификация HuggingFace
from huggingface_hub import login
import os

login(userdata.get('HF_TOKEN'))

In [ ]:
# 3. Загрузка чекпоинта Gemma-3-1B
!mkdir -p gemma3_1b_ckpt
# !gsutil -m cp -r gs://gemma-data/checkpoints/gemma3-1b-it ./gemma3_1b_ckpt

In [ ]:
import os
# 4. Конфигурация
config = {

    "dataset_repo": "veriga/openwebtext-gemma3-tokenized-1024",
    "output_dir": "./activations_layer23-64",
    "target_layer": 23,
    "max_seq_len": 1024,
    "batch_size": 64,          # BS=4 безопасно для T4 (16GB)
    "dtype": "bfloat16",      # JAX на GPU умеет эмулировать bf16
    "save_dtype": "bfloat16",
    "max_examples": None,     # None = весь датасет
    "resume": True,
    "push_activations": True,
    "activations_repo": "veriga/openwebtext-gemma3-tokenized-1024-activations-layer23",
    "upload_batch": 64,       # Делать коммит в HF каждые 64 шардов
    "upload_workers": 2,
    # Parquet-конвертация
    "parquet_batch_size": 10,      # Шардов на один Parquet-файл
    "parquet_compression": "zstd",
    "parquet_compression_level": 3,
}

os.makedirs(config["output_dir"], exist_ok=True)

In [ ]:
import json
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import jax
import jax.numpy as jnp
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from gemma import gm
from gemma.gm.nn import _modules
from datasets import load_dataset
from huggingface_hub import HfApi, CommitOperationAdd, create_commit

def _build_block_kwargs(config, layer_idx):
    attn_type = config.attention_types[layer_idx]
    is_local = attn_type == _modules.AttentionType.LOCAL_SLIDING
    return dict(
        num_heads=config.num_heads,
        num_kv_heads=config.num_kv_heads,
        embed_dim=config.embed_dim,
        head_dim=config.head_dim,
        hidden_dim=config.hidden_dim,
        sliding_window_size=config.sliding_window_size,
        use_post_attn_norm=config.use_post_attn_norm,
        use_post_ffw_norm=config.use_post_ffw_norm,
        attn_logits_soft_cap=config.attn_logits_soft_cap,
        attn_type=attn_type,
        query_pre_attn_scalar=config.query_pre_attn_scalar(),
        transpose_gating_einsum=config.transpose_gating_einsum,
        use_qk_norm=config.use_qk_norm,
        rope_base_frequency=config.local_base_frequency if is_local else config.global_base_frequency,
        rope_scale_factor=config.local_scale_factor if is_local else config.global_scale_factor,
    )

def make_forward_to_layer(target_layer: int):
    # Ручной проход до нужного слоя, чтобы избежать бага return_hidden_states
    model = gm.nn.Gemma3_1B()
    config = model.config
    blocks = [_modules.Block(name=f'layer_{i}', **_build_block_kwargs(config, i)) for i in range(target_layer)]

    @jax.jit
    def forward_fn(params, tokens, masks):
        B, L = tokens.shape
        embedding_table = params['embedder']['input_embedding']
        x = embedding_table[tokens]
        x = x * jnp.sqrt(config.embed_dim).astype(x.dtype)

        positions = jnp.broadcast_to(jnp.arange(L)[None, :], (B, L))
        causal = jnp.tril(jnp.ones((L, L), dtype=jnp.bool_))
        attn_mask = causal[None, :, :] & masks[:, None, :].astype(jnp.bool_)

        for i, block in enumerate(blocks):
            _, x = block.apply({'params': params[f'layer_{i}']}, x, positions, None, attn_mask)

        return x
    return forward_fn

def dataset_to_batches(ds, batch_size, max_seq_len, max_examples=None):
    total = len(ds)
    if max_examples: total = min(total, max_examples)
    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        batch = ds[start:end]
        raw_tokens = batch.get("tokens") or batch.get("input_ids")

        batch_tokens, batch_masks = [], []
        for tokens in raw_tokens:
            t_arr = np.array(tokens[:max_seq_len], dtype=np.int32)
            orig_len = len(t_arr)
            pad_len = max_seq_len - orig_len
            if pad_len > 0: t_arr = np.pad(t_arr, (0, pad_len))
            m_arr = np.zeros(max_seq_len, dtype=np.int32)
            m_arr[:orig_len] = 1
            batch_tokens.append(t_arr)
            batch_masks.append(m_arr)
        yield np.stack(batch_tokens), np.stack(batch_masks)

def upload_shard_batch(shard_paths, repo_id, max_retries=3):
    ops = [CommitOperationAdd(path_in_repo=os.path.basename(p), path_or_fileobj=p) for p in shard_paths]
    for attempt in range(max_retries):
        try:
            create_commit(repo_id=repo_id, operations=ops,
                          commit_message=f"Add {len(shard_paths)} shards",
                          repo_type="dataset")
            return True
        except Exception as e:
            print(f"Upload error: {e}")
            time.sleep(10 * (2**attempt))
    return False

# Parquet schema (определяем один раз)
parquet_schema = pa.schema([
    pa.field("activations", pa.list_(pa.float32(), config["max_seq_len"] * 1152)),
    pa.field("mask", pa.list_(pa.int32(), config["max_seq_len"])),
    pa.field("tokens", pa.list_(pa.int32(), config["max_seq_len"])),
])

def shards_to_parquet(shard_paths, parquet_schema, config):
    """Конвертирует список npy-шардов в один Parquet-файл и возвращает путь."""
    all_act, all_mask, all_tok = [], [], []

    for sp in shard_paths:
        sid = os.path.basename(sp).split("shard_")[1].split(".npy")[0]
        act = np.array(jnp.load(sp), dtype=np.float32)
        mask = np.load(sp.replace(".npy", "_masks.npy")).astype(np.int32)
        tok = np.load(sp.replace(".npy", "_tokens.npy")).astype(np.int32)
        all_act.append(act)
        all_mask.append(mask)
        all_tok.append(tok)

    act_cat = np.concatenate(all_act, axis=0)
    mask_cat = np.concatenate(all_mask, axis=0)
    tok_cat = np.concatenate(all_tok, axis=0)
    del all_act, all_mask, all_tok

    N = act_cat.shape[0]
    table = pa.table({
        "activations": act_cat.reshape(N, -1).tolist(),
        "mask": mask_cat.tolist(),
        "tokens": tok_cat.tolist(),
    }, schema=parquet_schema)

    del act_cat, mask_cat, tok_cat
    return table

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print("🔧 Загрузка параметров...")
params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)
forward_fn = make_forward_to_layer(config["target_layer"])

api = HfApi()
parquet_schema = pa.schema([
    pa.field("activations", pa.list_(pa.float32(), config["max_seq_len"] * 1152)),
    pa.field("mask", pa.list_(pa.int32(), config["max_seq_len"])),
    pa.field("tokens", pa.list_(pa.int32(), config["max_seq_len"])),
])

print("📂 Загрузка датасета...")
ds = load_dataset(config["dataset_repo"],
                  split="train",
                  cache_dir="/content/drive/Shareddrives/.../hf_cache")

output_dir = config["output_dir"]
os.makedirs(output_dir, exist_ok=True)

meta_path = os.path.join(output_dir, "metadata.json")
start_shard = 0
if config["resume"] and os.path.exists(meta_path):
    with open(meta_path) as f:
        metadata = json.load(f)
    start_shard = metadata.get("next_shard", 0)
    print(f"▶️ Resuming from shard {start_shard}")

batch_gen = dataset_to_batches(ds, config["batch_size"], config["max_seq_len"], config["max_examples"])
if start_shard > 0:
    print(f"⏩ Skipping {start_shard} shards...")
    for _ in range(start_shard):
        next(batch_gen, None)

save_dtype = config["save_dtype"]
shard_idx = start_shard
t_start = time.time()
pending_npy = []      # NPY файлы, ожидающие пуша
pending_pq = []       # пути к npy, ожидающие конвертации в parquet
parquet_batch_size = config["parquet_batch_size"]

print(f"🚀 Начинаем с shard {shard_idx}")
for batch_tokens, batch_masks in batch_gen:
    hidden = forward_fn(params, jnp.array(batch_tokens), jnp.array(batch_masks))
    hidden_np = np.asarray(hidden) * batch_masks[:, :, None].astype(np.float32)

    paths = []
    for suffix, data in [("", hidden_np.astype(save_dtype)), ("_tokens", batch_tokens), ("_masks", batch_masks)]:
        path = os.path.join(config["output_dir"], f"shard_{shard_idx:06d}{suffix}.npy")
        np.save(path, data)
        paths.append(path)

    # Добавляем путь к основному шарду (без _masks/_tokens) в очередь parquet
    pending_pq.append(paths[0])

    # === NPY upload (оригинальный механизм) ===
    if config["push_activations"]:
        pending_npy.extend(paths)

    # === Parquet: конвертим батч и пушим ===
    if len(pending_pq) >= parquet_batch_size:
        parquet_batch_start = shard_idx - len(pending_pq) + 1
        parquet_batch_end = shard_idx

        print(f"📦 Converting shards {parquet_batch_start}-{parquet_batch_end} to Parquet...")
        table = shards_to_parquet(pending_pq, parquet_schema, config)

        parquet_name = f"train-{parquet_batch_start:06d}-{parquet_batch_end:06d}.parquet"
        parquet_path = os.path.join(output_dir, parquet_name)
        pq.write_table(table, parquet_path,
                      compression=config["parquet_compression"],
                      compression_level=config["parquet_compression_level"])
        del table

        size_mb = os.path.getsize(parquet_path) / 1e6
        print(f"  ✅ {parquet_name}: {size_mb:.0f} MB")

        # Пушим parquet
        try:
            api.upload_file(
                path_or_fileobj=parquet_path,
                path_in_repo=f"data/{parquet_name}",
                repo_id=config["activations_repo"],
                repo_type="dataset",
                commit_message=f"Add parquet batch {parquet_batch_start}-{parquet_batch_end}",
            )
            print(f"  📤 Pushed to Hub")
        except Exception as e:
            print(f"  ❌ Parquet upload failed: {e}")

        os.remove(parquet_path)
        pending_pq = []

    # === NPY: пушим батч ===
    if config["push_activations"] and len(pending_npy) >= config["upload_batch"] * 3:
        print(f"\n⏳ Uploading {len(pending_npy)} NPY files...")
        upload_shard_batch(pending_npy[:], config["activations_repo"])
        print("✅ Upload complete. Resuming processing...")
        pending_npy = []

    shard_idx += 1
    if shard_idx % 10 == 0:
        elapsed = time.time() - t_start
        print(f"  Shard {shard_idx} | {shard_idx*config['batch_size']/elapsed:.1f} ex/s | {elapsed:.1f}s")
        with open(meta_path, "w") as f: json.dump({"next_shard": shard_idx}, f)

# === Финализация: остатки ===
print("\n📦 Converting remaining shards to Parquet...")
if pending_pq:
    parquet_batch_start = shard_idx - len(pending_pq)
    parquet_batch_end = shard_idx - 1
    table = shards_to_parquet(pending_pq, parquet_schema, config)
    parquet_name = f"train-{parquet_batch_start:06d}-{parquet_batch_end:06d}.parquet"
    parquet_path = os.path.join(output_dir, parquet_name)
    pq.write_table(table, parquet_path,
                  compression=config["parquet_compression"],
                  compression_level=config["parquet_compression_level"])
    del table
    size_mb = os.path.getsize(parquet_path) / 1e6
    print(f"  ✅ {parquet_name}: {size_mb:.0f} MB")
    try:
        api.upload_file(
            path_or_fileobj=parquet_path,
            path_in_repo=f"data/{parquet_name}",
            repo_id=config["activations_repo"],
            repo_type="dataset",
            commit_message=f"Add final parquet batch {parquet_batch_start}-{parquet_batch_end}",
        )
        print(f"  📤 Pushed to Hub")
    except Exception as e:
        print(f"  ❌ Parquet upload failed: {e}")
    os.remove(parquet_path)

if pending_npy:
    print(f"\n⏳ Uploading {len(pending_npy)} remaining NPY files...")
    upload_shard_batch(pending_npy[:], config["activations_repo"])

with open(meta_path, "w") as f: json.dump({"next_shard": shard_idx}, f)
print(f"✅ Обработка завершена! Всего: {shard_idx} шардов")

In [ ]:
pending_npy

In [ ]:
upload_shard_batch(pending_npy[:], config["activations_repo"])

In [ ]:
upload_shard_batch([os.path.join(config["output_dir"], "metadata.json")], config["activations_repo"])